In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 120)

# %matplotlib inline

# Data loading

In [4]:
import glob

parquet_files = glob.glob('data/*.parquet')

df_list = []
for file in parquet_files:
    df = pd.read_parquet(file, engine='fastparquet')
    df_list.append(df)

if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
else:
    print("No se encontraron archivos .parquet en el directorio 'sample_data'.")


combined_df.describe()
combined_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 223549 entries, 0 to 223548
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             223549 non-null  object
 1   comment_text   223549 non-null  object
 2   toxic          223549 non-null  int64 
 3   severe_toxic   223549 non-null  int64 
 4   obscene        223549 non-null  int64 
 5   threat         223549 non-null  int64 
 6   insult         223549 non-null  int64 
 7   identity_hate  223549 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 13.6+ MB


# Cleaning and simplification of columns

In [5]:
# ID is not an relevant columns thats really describe some behavior of data.
relevant_columns = combined_df.columns.drop(['id'])
toxic_df = combined_df[relevant_columns]



# Some study
Esta data al ser enfocada a comentarios y clasificada por tipo de comentario. Asumimos que ninguna columna tiene valores `NaN`, incluso la misma columna principal `coment_text`.

In [6]:
toxic_df.shape, toxic_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 223549 entries, 0 to 223548
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   comment_text   223549 non-null  object
 1   toxic          223549 non-null  int64 
 2   severe_toxic   223549 non-null  int64 
 3   obscene        223549 non-null  int64 
 4   threat         223549 non-null  int64 
 5   insult         223549 non-null  int64 
 6   identity_hate  223549 non-null  int64 
dtypes: int64(6), object(1)
memory usage: 11.9+ MB


((223549, 7), None)

## Análisis de las variables booleanas

In [32]:
# Demostrando que ninguna columna tiene valores NaN
possible_NaN_in_data = np.array([
    toxic_df[column].isna().astype(int)
    for column in toxic_df.columns])
print(possible_NaN_in_data)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


Las variables booleanas representan en nivel de toxicidad de los comentarios.
En esta parte, observamos la gran mayoría (al menos el 90%) de los comentarios en este dataset, son tóxicos.

In [30]:
columnas_numericas = toxic_df.select_dtypes(include=['number']).columns
an_list = []
for column in columnas_numericas:
    true_values = toxic_df[column].sum()
    false_values = toxic_df[column].size - true_values
    true_percentage = true_values / toxic_df[column].size * 100
    an_list.append({
        'true_value': true_values,
        'false_value': false_values,
        'true_percentage': true_percentage
    })

dd = pd.DataFrame(an_list, index=columnas_numericas)
print(dd)


               true_value  false_value  true_percentage
toxic               21384       202165         9.565688
severe_toxic         1962       221587         0.877660
obscene             12140       211409         5.430577
threat                689       222860         0.308210
insult              11304       212245         5.056610
identity_hate        2117       221432         0.946996


Con esta parte, demostramos para todas las columnas booleanas excluyendo de `toxic`, siempre que están activas, siempre `toxic` está activo.

In [36]:
bool_columns = toxic_df.select_dtypes(include=['number']).columns.drop('toxic')
an_list = []
for column in columnas_numericas:
    true_values = toxic_df[column].sum()
    false_values = toxic_df['toxic'].sum()
    # true_percentage = true_values / toxic_df[column].size * 100
    an_list.append({
        'yo': true_values,
        'toxic': false_values,
        # 'true_percentage': true_percentage
    })

dd = pd.DataFrame(an_list, index=columnas_numericas)
print(dd)

                  yo  toxic
toxic          21384  21384
severe_toxic    1962  21384
obscene        12140  21384
threat           689  21384
insult         11304  21384
identity_hate   2117  21384
